# Problem 1. 다이아몬드 가격 예측하기

## 문제 개요
보석 감정 업체는 감정서 정보만으로 다이아몬드의 적정 판매가를 빠르게 산정하는 모델이 필요합니다.
캐럿, 컷, 색상, 투명도, 크기 정보를 이용해 **다이아몬드 가격(price, USD)** 을 예측하세요.

## 파일 구조
```
problem01/
├── work.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train.csv
    ├── test.csv
    └── sample_submission.csv
```

## 데이터
- `data/train.csv` : 40,000개 (price 포함)
- `data/test.csv` : 13,940개 (price 없음)

| 컬럼 | 설명 |
|---|---|
| id | 고유 번호 |
| carat | 무게(캐럿) |
| cut | 컷 등급: Fair < Good < Very Good < Premium < Ideal |
| color | 색상 등급: J(최하) < I < H < G < F < E < D(최상) |
| clarity | 투명도: I1(최하) < SI2 < SI1 < VS2 < VS1 < VVS2 < VVS1 < IF(최상) |
| depth | 깊이 비율(%) = 100 × z / 평균(x, y) |
| table | 상단 평면 폭 비율(%) |
| x, y, z | 가로·세로·높이(mm) |
| **price** | **가격(USD), 예측 대상** |

### 데이터 특징
- 실제 수집 데이터처럼 **결측값과 표기 오류**가 섞여 있습니다. 범주 값의 표기를 꼼꼼히 확인하세요.
- 일부 크기 값(x, y, z)은 측정 오류로 0이 기록되어 있습니다.
- 출처: ggplot2 diamonds 데이터셋을 대회용으로 재구성

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`problem01/`)에 **`submission.csv`** 로 저장

## 평가
**RMSLE (Root Mean Squared Logarithmic Error)** — 낮을수록 좋습니다.

$$\text{RMSLE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(\log(1+\hat{y}_i) - \log(1+y_i)\right)^2}$$

- 음수 예측값은 0으로 바꿔서 계산합니다.
- 비싼 다이아몬드의 큰 오차보다 **비율 오차**가 중요한 지표입니다.

## 제출 형식
`submission.csv`

| id | price |
|---|---|
| 40000 | 3500.0 |
| 40001 | 812.5 |

- 컬럼명 `id`, `price`, 테스트의 모든 id 포함
- 포맷이 틀리면 0점 처리됩니다.

## 제출 검증 규칙
- 테스트의 모든 ID를 정확히 한 번씩 제출합니다. 누락·중복·추가 ID는 오류이며, 행 순서는 자유입니다(채점 시 ID로 정렬합니다).
- CSV는 UTF-8(또는 UTF-8 BOM), 쉼표 구분, 헤더 포함 형식입니다. 아래 두 컬럼만 제출하며 컬럼 순서는 자유입니다.
- 비어 있는 셀, NaN, 무한대, 깨진 CSV는 오류입니다. 올바르지 않은 제출은 해당 문제 0점과 ERROR로 기록합니다.
- 최종 파일은 이 문제 폴더의 `submission.csv`입니다. 다른 이름이나 하위 폴더 파일은 수집하지 않습니다.
- 컬럼: `id,price`; id는 테스트 CSV에 있는 정수 ID, price는 유한한 실수(USD)입니다. 원문 규칙대로 음수 예측은 RMSLE 계산에서 0으로 클리핑합니다. 가격의 상한은 제한하지 않습니다.
- 행 수: 13,940.

## 100점 환산
원본 정답 노트의 제공 베이스라인 RMSLE **0.3938**을 기준오차로 사용합니다.

`score = 100 × max(0, 1 − RMSLE / 0.3938)`

정답과 같으면 100점, RMSLE가 0.3938 이상이면 0점입니다. RMSLE가 낮을수록 점수가 높습니다. 세 문제 점수 합계는 300점입니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
DATA_DIR = Path("data")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
print("train:", train.shape, "test:", test.shape)
train.head()

## TODO
데이터를 확인하고 전처리·모델·검증 방법을 구현하세요. 아래는 원문에서 제공한 간단한 베이스라인입니다. 필요에 맞게 수정하세요.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from sklearn.model_selection import KFold
from sklearn.ensemble import ExtraTreesRegressor
from catboost import CatBoostRegressor                                      # 없으면: !pip install catboost

CUT = ["Fair", "Good", "Very Good", "Premium", "Ideal"]; COL = list("JIHGFED")
CLA = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]            # 문제에 주어진 등급 순서(낮음 -> 높음)

def clean(df, cat_as_str=False):
    X = df.drop(columns=["id", "price"], errors="ignore").copy()
    # 표기 오류 정리: 대소문자(ideal/Ideal), 끝 공백(VS2 ), 순서형 인코딩
    X["cut"] = X.cut.str.strip().str.title().map({c: i for i, c in enumerate(CUT)})
    X["color"] = X.color.str.strip().str.upper().map({c: i for i, c in enumerate(COL)})
    X["clarity"] = X.clarity.str.strip().str.upper().map({c: i for i, c in enumerate(CLA)})
    for c in "xyz": X.loc[X[c] <= 0, c] = np.nan                           # 측정 오류 0 -> 결측
    X.loc[X.y > 20, "y"] = np.nan; X.loc[X.z > 12, "z"] = np.nan           # 비정상 큰 값(예: y=58.9)
    ok = X[["x", "y", "z"]].notna().all(axis=1)                            # x,y,z 모두 없는 행은 캐럿으로 추정(carat ∝ 부피)
    for c in "xyz":
        a = np.polyfit(np.log(X.loc[ok, "carat"]), np.log(X.loc[ok, c]), 1)
        m = X[c].isna() & X[["x", "y", "z"]].isna().all(axis=1)
        X.loc[m, c] = np.exp(np.polyval(a, np.log(X.loc[m, "carat"])))
    xy = X[["x", "y"]].mean(axis=1)
    X["depth"] = X.depth.fillna(100 * X.z / xy)                            # depth = 100*z/평균(x,y) 로 복원
    X["z"] = X.z.fillna(X.depth * xy / 100)
    X["x"] = X.x.fillna(X.y); X["y"] = X.y.fillna(X.x)
    X["vol"] = X.x * X.y * X.z; X["log_carat"] = np.log(X.carat); X["xy_ratio"] = X.x / X.y
    X["carat_cut"] = X.carat * (X.cut + 1); X["quality"] = X.cut + X.color + X.clarity
    X["carat_clar"] = X.carat * (X.clarity + 1); X["carat_col"] = X.carat * (X.color + 1)
    if cat_as_str:                                                          # CatBoost 범주형 직접 입력용
        for c, L in (("cut", CUT), ("color", COL), ("clarity", CLA)): X[c] = X[c].astype(int).map(dict(enumerate(L)))
    return X

X, X_test = clean(train), clean(test)
Xs, Xs_test = clean(train, True), clean(test, True)
assert list(X.columns) == list(X_test.columns) and X[["cut", "color", "clarity"]].notna().all().all() and X_test[["cut", "color", "clarity"]].notna().all().all()
y = np.log1p(train.price.values)                                           # RMSLE = log1p 공간의 RMSE
FILL = X.median()                                                          # ExtraTrees 용 결측 대치(학습 중앙값)
CATS = ["cut", "color", "clarity"]
cb = lambda seed, cat: CatBoostRegressor(iterations=1500, learning_rate=0.06, depth=8, l2_leaf_reg=3, random_seed=seed, verbose=0,
                                         thread_count=-1, cat_features=CATS if cat else None)
et = lambda seed: ExtraTreesRegressor(500, min_samples_leaf=1, max_features=0.8, n_jobs=-1, random_state=seed)
rmsle = lambda p, t: float(np.sqrt(np.mean((p - t) ** 2)))

# 타깃 변환(v3 개선): log1p(price)에서 캐럿 추세 a*log(carat)+b 를 빼고 잔차를 학습한 뒤 더해서 복원 (큰 돌 구간의 외삽 보완)
def fit_offset(Xtr, ytr):
    co = np.polyfit(np.log(Xtr.carat.values), ytr, 1)
    return lambda Z: np.polyval(co, np.log(Z.carat.values))
def cb_off_fit_predict(seed, Xa, ya, Xb):
    f = fit_offset(Xa, ya)
    return cb(seed, False).fit(Xa, ya - f(Xa)).predict(Xb) + f(Xb)

# ---- 5-Fold CV: 빠른 모델(cb_off, et)만 노트북에서 재현. cb_cat(CV 약 7분)과 3모델 블렌드는 실험으로 측정한 값을 기록 ----
oof = {"cb_off": np.zeros(len(X)), "et": np.zeros(len(X))}
for a, b in KFold(5, shuffle=True, random_state=42).split(X):
    oof["cb_off"][b] = cb_off_fit_predict(0, X.iloc[a], y[a], X.iloc[b])
    oof["et"][b] = et(0).fit(X.iloc[a].fillna(FILL), y[a]).predict(X.iloc[b].fillna(FILL))
for k in oof: print(f"{k} CV RMSLE: {rmsle(oof[k], y):.4f}")
print(f"cb_off+et 평균 CV RMSLE: {rmsle(np.mean(list(oof.values()), 0), y):.4f}")
print("(실험 기록) cb_off+cb_cat+et 평균 0.0810 -> 환산 점수 약 79.4 | 이전 v2 0.0814 (cb_ord+cb_cat+et), v1 0.0831 (HGB+RF)")

# ---- 전체 train 재학습 후 test 예측: cb_off(3 seed) + cb_cat + ExtraTrees 를 같은 비중(1/3)으로 평균 ----
p_off = np.mean([cb_off_fit_predict(s, X, y, X_test) for s in range(3)], axis=0)
p_cat = cb(0, True).fit(Xs, y).predict(Xs_test)
p_et = np.mean([et(s).fit(X.fillna(FILL), y).predict(X_test.fillna(FILL)) for s in range(2)], axis=0)
pred = np.clip(np.expm1((p_off + p_cat + p_et) / 3), 0, None)               # log1p 역변환, 음수는 0

In [ ]:
submission = pd.DataFrame({"id": test["id"], "price": pred})
submission.to_csv("submission.csv", index=False)

In [ ]:
submission = pd.read_csv("submission.csv")
print(submission.shape)
assert len(submission) == len(test)
assert submission["id"].is_unique
assert set(submission["id"]) == set(test["id"])
submission.head()
assert list(submission.columns) == ["id", "price"] and submission.price.notna().all() and np.isfinite(submission.price).all() and (submission.price >= 0).all()